# Q4: Defect Risk Classification
## Can we flag high-defect-risk suppliers before the order is placed?

**Report Section:** Question 4 · Classification  
**Data Source:** Supply Chain ML Predictive Modelling Report

## Objective

- Engineer a binary risk label from continuous `defect_rate` and train a classifier to predict it from pre-order features only
- Optimize decision threshold to balance recall (catch high-risk suppliers) vs. precision (avoid false alarms)
- Create a supplier risk scorecard that procurement can use as a deployment-ready watchlist

In [ ]:
# Step 0: Setup & Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc, 
    precision_recall_curve, roc_auc_score
)

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load dataset
# df = pd.read_csv('supply_chain_data.csv')  # Replace with actual data path
# print(df.head())
# print(df.info())
# print(df['defect_rate'].describe())

## Implementation Plan

### Step 1: Engineer the Risk Label from Defect Rate
- Convert continuous `defect_rate` to binary: high_defect_risk = 1 if defect_rate > median(defect_rate), else 0
- Plot defect_rate distribution with threshold marked
- Count class balance; median threshold ensures 50/50 split by construction

### Step 2: Feature Selection — Pre-Order Risk Signals
- Use ONLY features available BEFORE an order is placed
- Exclude post-order features (return_rate) to ensure deployable model
- Include: quality_score, supplier_reliability_score, lead_time_variance, on_time_delivery_rate, price_per_unit, delivery_mode, delivery_term_code, payment_term_days, offer_validity_days, order_frequency_monthly, items_offered, items_requested, temporal_month

### Step 3: Gradient Boosting Classifier — Primary Model
- Train GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, random_state=42)
- Evaluate: classification_report, ROC-AUC, confusion matrix heatmap
- Maximize recall on class 1 (high risk) — one missed defect supplier can exceed model development cost

### Step 4: Threshold Optimisation — Precision-Recall Trade-off
- Use precision_recall_curve to plot precision vs. recall across all thresholds
- Identify threshold that maximises F1 and threshold that maximises recall at >70% precision
- Present both thresholds as business decision (aggressive vs. conservative)

### Step 5: Risk Score Distribution — Supplier Portfolio View
- Use predict_proba() to assign continuous defect risk score (0–1) to every supplier
- Plot overlapping histogram of risk scores split by actual high_defect_risk class
- Assess model separation quality; overlapping = uncertain

### Step 6: Risk Segmentation by Delivery Mode & Procurement Action
- Group predicted risk scores by delivery_mode and procurement_action_code
- Plot heatmap: rows = delivery mode, columns = procurement action, cells = mean risk
- Highlight highest-risk combination

### Step 7: Supplier Risk Scorecard — Executive Output
- Build final supplier table: quality_score, supplier_reliability_score, defect_rate, on_time_delivery_rate, predicted_risk_score, risk_flag (High/Low)
- Sort by predicted_risk_score descending
- Colour-code risk_flag; this is the boardroom deliverable

In [ ]:
# STEP 1: Engineer the Risk Label from Defect Rate

# # Create binary risk label
# defect_median = df['defect_rate'].median()
# df['high_defect_risk'] = (df['defect_rate'] > defect_median).astype(int)

# # Check class distribution
# risk_dist = df['high_defect_risk'].value_counts()
# risk_pct = df['high_defect_risk'].value_counts(normalize=True) * 100

# print(f"Defect Rate Threshold (Median): {defect_median:.4f}")
# print(f"\nRisk Class Distribution:")
# print(f"  Low Risk (0): {risk_dist.get(0, 0)} ({risk_pct.get(0, 0):.1f}%)")
# print(f"  High Risk (1): {risk_dist.get(1, 0)} ({risk_pct.get(1, 0):.1f}%)")

In [ ]:
# STEP 1 Visualization: Defect Rate Distribution with Threshold

# fig, ax = plt.subplots(figsize=(10, 6))
# ax.hist(df['defect_rate'], bins=40, edgecolor='black', alpha=0.7, color='steelblue')
# ax.axvline(x=defect_median, color='red', linestyle='--', linewidth=2, label=f'Threshold (Median = {defect_median:.4f})')
# ax.set_xlabel('Defect Rate')
# ax.set_ylabel('Frequency')
# ax.set_title('Defect Rate Distribution with Risk Threshold')
# ax.legend()
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 2: Feature Selection — Pre-Order Risk Signals

# # Define features available BEFORE order placement
# numerical_features = [
#     'quality_score', 'supplier_reliability_score', 'lead_time_variance',
#     'on_time_delivery_rate', 'price_per_unit', 'payment_term_days',
#     'offer_validity_days', 'order_frequency_monthly', 'items_offered', 'items_requested'
# ]
# categorical_features = ['delivery_mode', 'delivery_term_code', 'temporal_month']

# # Note: Exclude post-order features like 'return_rate'
# all_features = numerical_features + categorical_features

# X = df[all_features]
# y = df['high_defect_risk']

# print("Features used for defect risk prediction (pre-order only):")
# print("  Numerical:", numerical_features)
# print("  Categorical:", categorical_features)
# print(f"\nTotal features: {len(all_features)}")

In [ ]:
# STEP 2 Visualization: Feature Availability Matrix

# feature_types = pd.DataFrame({
#     'Feature': all_features,
#     'Type': ['Numerical'] * len(numerical_features) + ['Categorical'] * len(categorical_features),
#     'Available_Pre_Order': ['Yes'] * len(all_features)  # All features we selected are pre-order
# })

# print("\nFeature Pre/Post Order Availability:")
# print(feature_types.to_string(index=False))

In [ ]:
# STEP 3: Gradient Boosting Classifier — Primary Model

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# # Create preprocessing pipeline
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', StandardScaler(), numerical_features),
#         ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
#     ])

# # Gradient Boosting Classifier
# gbc_pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model', GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=3, random_state=42))
# ])

# gbc_pipeline.fit(X_train, y_train)
# y_pred_gbc = gbc_pipeline.predict(X_test)
# y_pred_proba_gbc = gbc_pipeline.predict_proba(X_test)[:, 1]

# # Classification report
# print("Gradient Boosting Classifier - Classification Report:")
# print(classification_report(y_test, y_pred_gbc, target_names=['Low Risk', 'High Risk']))

# # AUC score
# roc_auc_gbc = roc_auc_score(y_test, y_pred_proba_gbc)
# print(f"\nROC-AUC Score: {roc_auc_gbc:.3f}")

In [ ]:
# STEP 3 Visualization: Confusion Matrix & ROC Curve

# cm_gbc = confusion_matrix(y_test, y_pred_gbc)
# fpr_gbc, tpr_gbc, _ = roc_curve(y_test, y_pred_proba_gbc)

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Confusion matrix
# sns.heatmap(cm_gbc, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar_kws={'label': 'Count'})
# axes[0].set_xlabel('Predicted')
# axes[0].set_ylabel('Actual')
# axes[0].set_title('Confusion Matrix - Defect Risk Classification')
# axes[0].set_xticklabels(['Low Risk', 'High Risk'])
# axes[0].set_yticklabels(['Low Risk', 'High Risk'])

# # ROC curve
# axes[1].plot(fpr_gbc, tpr_gbc, label=f'GBC (AUC = {roc_auc_gbc:.3f})', linewidth=2, color='blue')
# axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline')
# axes[1].set_xlabel('False Positive Rate')
# axes[1].set_ylabel('True Positive Rate')
# axes[1].set_title('ROC Curve - Defect Risk Classification')
# axes[1].legend(loc='lower right')
# axes[1].grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 4: Threshold Optimisation — Precision-Recall Trade-off

# precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba_gbc)

# # Calculate F1 scores
# f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)

# # Find optimal thresholds
# optimal_f1_idx = np.argmax(f1_scores)
# optimal_f1_threshold = thresholds[optimal_f1_idx]
# optimal_f1 = f1_scores[optimal_f1_idx]

# # Find threshold that maximizes recall at >70% precision
# high_precision_mask = precision[:-1] > 0.70
# if high_precision_mask.any():
#     high_precision_idx = np.argmax(recall[:-1][high_precision_mask])
#     conservative_threshold = thresholds[:-1][high_precision_mask][high_precision_idx]
#     conservative_recall = recall[:-1][high_precision_mask][high_precision_idx]
#     conservative_precision = precision[:-1][high_precision_mask][high_precision_idx]
# else:
#     conservative_threshold = optimal_f1_threshold
#     conservative_recall = recall[optimal_f1_idx]
#     conservative_precision = precision[optimal_f1_idx]

# print("\nThreshold Optimisation Results:")
# print(f"\n1. F1-Maximizing Threshold: {optimal_f1_threshold:.3f}")
# print(f"   Precision: {precision[optimal_f1_idx]:.3f}, Recall: {recall[optimal_f1_idx]:.3f}, F1: {optimal_f1:.3f}")
# print(f"\n2. Conservative Threshold (>70% Precision): {conservative_threshold:.3f}")
# print(f"   Precision: {conservative_precision:.3f}, Recall: {conservative_recall:.3f}")
# print(f"\n→ Use aggressive threshold ({optimal_f1_threshold:.3f}) to catch more high-risk suppliers")
# print(f"→ Use conservative threshold ({conservative_threshold:.3f}) to reduce false positives")

In [ ]:
# STEP 4 Visualization: Precision-Recall Curve

# fig, ax = plt.subplots(figsize=(10, 7))
# ax.plot(recall[:-1], precision[:-1], linewidth=2, color='blue', label='Precision-Recall Curve')
# ax.scatter([recall[optimal_f1_idx]], [precision[optimal_f1_idx]], color='red', s=100, 
#            label=f'F1-Max: threshold={optimal_f1_threshold:.3f}', zorder=5)
# ax.scatter([conservative_recall], [conservative_precision], color='orange', s=100,
#            label=f'Conservative (>70% Prec): threshold={conservative_threshold:.3f}', zorder=5)
# ax.axhline(y=0.70, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='70% Precision Line')
# ax.set_xlabel('Recall')
# ax.set_ylabel('Precision')
# ax.set_title('Precision-Recall Curve: Threshold Optimisation')
# ax.legend(loc='best')
# ax.grid(True, alpha=0.3)
# ax.set_xlim([0, 1])
# ax.set_ylim([0, 1])
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 5: Risk Score Distribution — Supplier Portfolio View

# # Create risk score dataframe
# risk_scores = y_pred_proba_gbc

# # Count high-risk suppliers
# high_risk_count = np.sum(risk_scores > optimal_f1_threshold)
# high_risk_pct = (high_risk_count / len(risk_scores)) * 100

# print(f"\nSuppliers Flagged as High-Risk (threshold={optimal_f1_threshold:.3f}):")
# print(f"  Count: {high_risk_count} out of {len(risk_scores)}")
# print(f"  Percentage: {high_risk_pct:.1f}%")

# # Separation quality
# low_risk_actual = risk_scores[y_test == 0]
# high_risk_actual = risk_scores[y_test == 1]

# print(f"\nModel Separation Quality:")
# print(f"  Mean score (actual low risk):  {low_risk_actual.mean():.3f} ± {low_risk_actual.std():.3f}")
# print(f"  Mean score (actual high risk): {high_risk_actual.mean():.3f} ± {high_risk_actual.std():.3f}")

In [ ]:
# STEP 5 Visualization: Risk Score Distribution (Overlapping Histograms)

# fig, ax = plt.subplots(figsize=(10, 6))
# ax.hist(low_risk_actual, bins=30, alpha=0.6, label='Actual Low Risk', color='green', edgecolor='black')
# ax.hist(high_risk_actual, bins=30, alpha=0.6, label='Actual High Risk', color='red', edgecolor='black')
# ax.axvline(x=optimal_f1_threshold, color='blue', linestyle='--', linewidth=2, label=f'Decision Threshold ({optimal_f1_threshold:.3f})')
# ax.set_xlabel('Predicted Defect Risk Score')
# ax.set_ylabel('Frequency')
# ax.set_title('Risk Score Distribution: Separation Quality')
# ax.legend()
# ax.grid(axis='y', alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 6: Risk Segmentation by Delivery Mode & Procurement Action

# # Create results dataframe
# X_test_reset = X_test.reset_index(drop=True)
# results_df = X_test_reset.copy()
# results_df['predicted_risk_score'] = y_pred_proba_gbc
# results_df['actual_high_risk'] = y_test.values

# # Create pivot table
# risk_matrix = results_df.pivot_table(
#     values='predicted_risk_score',
#     index='delivery_mode',
#     columns='procurement_action_code',
#     aggfunc='mean'
# )

# print("\nMean Predicted Defect Risk Score by Delivery Mode × Procurement Action:")
# print(risk_matrix)

# # Identify highest-risk combination
# max_risk = risk_matrix.values.max()
# max_idx = np.unravel_index(risk_matrix.values.argmax(), risk_matrix.shape)
# max_mode = risk_matrix.index[max_idx[0]]
# max_action = risk_matrix.columns[max_idx[1]]

# print(f"\n⚠️ Highest-Risk Combination: {max_mode} × {max_action} (Risk Score: {max_risk:.3f})")
# print("   Recommendation: Avoid this route-strategy pair for quality-sensitive orders.")

In [ ]:
# STEP 6 Visualization: Risk Heatmap

# fig, ax = plt.subplots(figsize=(12, 6))
# sns.heatmap(risk_matrix, annot=True, fmt='.3f', cmap='RdYlGn_r', center=risk_matrix.values.mean(),
#             ax=ax, cbar_kws={'label': 'Mean Predicted Risk Score'}, linewidths=1, linecolor='gray')
# ax.set_title('Mean Predicted Defect Risk Score by Delivery Mode × Procurement Action')
# ax.set_xlabel('Procurement Action Code')
# ax.set_ylabel('Delivery Mode')
# plt.tight_layout()
# plt.show()

In [ ]:
# STEP 7: Supplier Risk Scorecard — Executive Output

# # Build supplier scorecard
# scorecard_df = df.iloc[X_test.index].copy()
# scorecard_df['predicted_risk_score'] = y_pred_proba_gbc
# scorecard_df['risk_flag'] = scorecard_df['predicted_risk_score'].apply(
#     lambda x: 'High Risk' if x > optimal_f1_threshold else 'Low Risk'
# )

# # Select columns for executive view
# scorecard_display = scorecard_df[[
#     'quality_score', 'supplier_reliability_score', 'defect_rate', 
#     'on_time_delivery_rate', 'predicted_risk_score', 'risk_flag'
# ]].sort_values('predicted_risk_score', ascending=False).reset_index(drop=True)

# print("\n" + "="*80)
# print("SUPPLIER RISK SCORECARD (Executive Summary)")
# print("="*80)
# print(scorecard_display.head(15).to_string())
# print(f"\n... and {max(0, len(scorecard_display) - 15)} more suppliers")
# print("\n" + "="*80)

In [ ]:
# STEP 7 Visualization: Styled Risk Scorecard (Top 20 High-Risk Suppliers)

# # Display top 20 for visualization
# top_risk = scorecard_display.head(20).copy()

# # Create a styled visualization
# fig, ax = plt.subplots(figsize=(14, 10))
# ax.axis('tight')
# ax.axis('off')

# # Create table
# table = ax.table(
#     cellText=top_risk.round(3).values,
#     colLabels=top_risk.columns,
#     cellLoc='center',
#     loc='center',
#     colWidths=[0.14, 0.16, 0.12, 0.16, 0.16, 0.14]
# )

# table.auto_set_font_size(False)
# table.set_fontsize(9)
# table.scale(1, 1.5)

# # Colour-code risk flags
# for i, risk_flag in enumerate(top_risk['risk_flag'].values, start=1):
#     if 'High' in risk_flag:
#         table[(i, 5)].set_facecolor('#ffcccc')  # Light red
#     else:
#         table[(i, 5)].set_facecolor('#ccffcc')  # Light green

# # Header styling
# for i in range(len(top_risk.columns)):
#     table[(0, i)].set_facecolor('#4472C4')
#     table[(0, i)].set_text_props(weight='bold', color='white')

# plt.title('Supplier Risk Scorecard: Top 20 High-Risk Suppliers', fontsize=14, fontweight='bold', pad=20)
# plt.tight_layout()
# plt.show()

## Key Takeaways

Once all steps are complete, your notebook will deliver:

1. **Risk Label Engineering** — Binary threshold from continuous defect_rate (median ensures 50/50 balance)
2. **Pre-Order Feature Set** — Validated exclusion of post-order features to ensure deployment feasibility
3. **Model Performance** — GBR classification report with focus on recall (catch high-risk), ROC-AUC
4. **Threshold Decisions** — Two competing thresholds presented as business decision: aggressive (high recall) vs. conservative (high precision)
5. **Portfolio Risk View** — Overlapping histograms show model separation quality; well-separated = deployable
6. **Operational Risk Map** — Heatmap reveals highest-risk delivery mode × procurement action pair; highlights single policy change
7. **Executive Watchlist** — Colour-coded supplier risk scorecard ranked by predicted risk; ready for immediate use

This is the end product: **a CEO does not need to understand gradient boosting — they need a ranked list of suppliers to scrutinise before the next procurement cycle.**